# AI Data Analytics Notebook
This notebook loads the sample CSV, shows basic statistics, and runs a Holt-Winters forecast using statsmodels.

In [ ]:
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing

df = pd.read_csv('../data/stock_aapl_sample.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.rename(columns={'Date':'ds','Close':'y'})
df = df[['ds','y']].dropna()
df = df.set_index('ds')
print(df.tail())
# Resample to daily if needed and interpolate
try:
    inferred = pd.infer_freq(df.index)
except Exception:
    inferred = None
if inferred is None:
    ts = df['y'].resample('D').mean().interpolate()
else:
    ts = df['y'].asfreq(inferred).interpolate()

seasonal_periods = 7 if len(ts) >= 14 else None
if seasonal_periods:
    model = ExponentialSmoothing(ts, trend='add', seasonal='add', seasonal_periods=seasonal_periods)
else:
    model = ExponentialSmoothing(ts, trend='add', seasonal=None)
fit = model.fit(optimized=True)
future = fit.forecast(30)
print(future.tail())
